<a href="https://colab.research.google.com/github/andrewtran117/MessingWithSafety/blob/main/firstprobe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

My first attempt at simple probing. With some comments for notes!

In [ ]:
!pip install -q transformers datasets scikit-learn

In [ ]:
from datasets import load_dataset

# SST-2: binary sentiment dataset (0 = negative, 1 = positive)
dataset = load_dataset("glue", "sst2")

# Use a subset so Colab doesn't explode
train_data = dataset["train"].shuffle(seed=42).select(range(2000))
test_data = dataset["validation"].shuffle(seed=42).select(range(500))

print(train_data[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

{'sentence': 'klein , charming in comedies like american pie and dead-on in election , ', 'label': 1, 'idx': 32326}


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_hidden_states=True)
model.eval()

# output_hidden_states=True reveals internal activations

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [ ]:
# check number layers
len(model.encoder.layer)

12

In [ ]:
def get_representation(texts, layer_idx=8, use_mean_pool=True):
    all_features = []

    for text in texts:
        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        with torch.no_grad():
            outputs = model(**enc)
            hidden_states = outputs.hidden_states  # tuple: (layer0,...,layerN)

        # hidden_states[layer_idx]: [batch_size, seq_len, hidden_dim]
        hs = hidden_states[layer_idx][0]  # first in batch: [seq_len, hidden_dim]

        if use_mean_pool:
            vec = hs.mean(dim=0)  # mean over tokens
        else:
            vec = hs[0]  # CLS token

        all_features.append(vec.numpy())

    return np.stack(all_features)

In [ ]:
train_data = dataset["train"].shuffle(seed=42).select(range(500))
test_data = dataset["validation"].shuffle(seed=42).select(range(200))

In [ ]:
train_texts = [ex["sentence"] for ex in train_data]
train_labels = [ex["label"] for ex in train_data]

test_texts = [ex["sentence"] for ex in test_data]
test_labels = [ex["label"] for ex in test_data]

# X_train = get_representation(train_texts, layer_idx=8)
X_train = get_representation(train_texts, layer_idx=11)
X_test = get_representation(test_texts, layer_idx=11)

y_train = np.array(train_labels)
y_test = np.array(test_labels)

X_train.shape, X_test.shape

((500, 768), (200, 768))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Linear probe accuracy:", acc)

Linear probe accuracy: 0.81
